# Preprocess cholera outbreaks

This notebook cleans and transforms the previously extracted and cleaned cholera outbreaks data from the [Integrated Disease Surveillance Programme of India](https://idsp.nic.in/) (IDSP) and saves the preprocessed cholera outbreaks in shapefile format.

Prior to running this notebook, execute
- `1_download_the_data/download_outbreaks.py` to download the outbreaks PDF files
- and `2_preprocess_the_data/extract_and_clean_cholera_outbreaks.py` to extract and clean cholera outbreaks from the downloaded outbreaks PDF files.

Properly extracting data from PDF files is tricky. In the present case it works for 91%. The remaining 9% need to be cleaned manually. Districts and start dates also need some more cleaning.

## Setup

In [ ]:
import re

import geopandas as gpd
import numpy as np
import pandas as pd

In [ ]:
pd.set_option("display.max_rows", None)

## Load and inspect cholera outbreaks

In [ ]:
cholera_outbreaks = pd.read_csv("../../../data/outbreaks/cholera_outbreaks.csv")
cholera_outbreaks.shape

In [ ]:
cholera_outbreaks.info()

In [ ]:
cholera_outbreaks.head()

In [ ]:
cholera_outbreaks.isna().sum()

In [ ]:
cholera_outbreaks[cholera_outbreaks.duplicated()].shape

## Check which columns need further cleaning

In [ ]:
# check cholera occurrences per column
for column in cholera_outbreaks.columns:
    cholera_occurrences = cholera_outbreaks[
        cholera_outbreaks[column].fillna("").str.contains("cholera")
    ].shape[0]
    print(f"Number of cholera occurrences in column {column}: {cholera_occurrences}")

In [ ]:
cholera_outbreaks_column_district = cholera_outbreaks[
    cholera_outbreaks["district"].str.contains("cholera")
]
cholera_outbreaks_column_district.shape

In [ ]:
cholera_outbreaks_column_cases = cholera_outbreaks[
    cholera_outbreaks["cases"].fillna("").str.contains("cholera")
]
cholera_outbreaks_column_cases.shape

In [ ]:
cholera_outbreaks_column_start_date = cholera_outbreaks[
    cholera_outbreaks["start_date"].fillna("").str.contains("cholera")
]
cholera_outbreaks_column_start_date.shape

In [ ]:
cholera_outbreaks_missing_district = cholera_outbreaks[
    cholera_outbreaks["district"] == "district missing"
]
cholera_outbreaks_missing_district.shape

Cholera outbreaks with missing district and start dates need to be cleaned manually.

In [ ]:
cholera_outbreaks = cholera_outbreaks[
    ~(cholera_outbreaks["district"].str.contains("cholera"))
    & ~(cholera_outbreaks["cases"].fillna("").str.contains("cholera"))
    & ~(cholera_outbreaks["start_date"].fillna("").str.contains("cholera"))
    & (cholera_outbreaks["district"] != "district missing")
].reset_index(drop=True)
cholera_outbreaks.shape

## Clean start dates

In [ ]:
# copy start date column
cholera_outbreaks["start_date_clean"] = cholera_outbreaks["start_date"]

In [ ]:
# fill na
cholera_outbreaks["start_date_clean"] = cholera_outbreaks["start_date_clean"].fillna("")

In [ ]:
# remove letters
cholera_outbreaks["start_date_clean"] = cholera_outbreaks["start_date_clean"].apply(
    lambda x: re.sub(r"[^0-9\s./-]+", "", x)
)

In [ ]:
# replace 0
cholera_outbreaks["start_date_clean"] = cholera_outbreaks["start_date_clean"].apply(
    lambda x: "" if x == "0" else x
)

In [ ]:
# separate two dates in a cell
cholera_outbreaks["start_date_clean"] = cholera_outbreaks["start_date_clean"].apply(
    lambda x: x.split(" ")[0] if len(x.split(" ")) == 2 else x
)

In [ ]:
# remove whitespace
cholera_outbreaks["start_date_clean"] = cholera_outbreaks["start_date_clean"].str.strip()

In [ ]:
cholera_outbreaks_missing_dates = cholera_outbreaks[cholera_outbreaks["start_date_clean"] == ""]
cholera_outbreaks_missing_dates.shape

Cholera outbreaks with missing start dates need to be cleaned manually before they can be further processed.

In [ ]:
# load and transform manually cleaned cholera outbreaks
cholera_outbreaks_manually_cleaned = pd.read_csv(
    "../../data/outbreaks/cholera_outbreaks_cleaned_manually.csv"
)
cholera_outbreaks_manually_cleaned["district"] = cholera_outbreaks_manually_cleaned[
    "district"
].str.lower()
cholera_outbreaks_manually_cleaned = cholera_outbreaks_manually_cleaned.rename(
    columns={"start_date": "start_date_clean"}
)
cholera_outbreaks_manually_cleaned = cholera_outbreaks_manually_cleaned.drop_duplicates()
cholera_outbreaks_manually_cleaned.shape

In [ ]:
# get cholera outbreaks without missing start dates
cholera_outbreaks = cholera_outbreaks[cholera_outbreaks["start_date_clean"] != ""].reset_index(
    drop=True
)
cholera_outbreaks.shape

In [ ]:
# merge cholera outbreaks again
cholera_outbreaks = pd.concat(
    [cholera_outbreaks, cholera_outbreaks_manually_cleaned], ignore_index=True
)
cholera_outbreaks.shape

In [ ]:
# harmonise date formats to extract year
cholera_outbreaks["year"] = cholera_outbreaks["start_date_clean"].apply(
    lambda x: (
        x.split(".")[2]
        if "." in x
        else (x.split("/")[2] if "/" in x else (x.split("-")[2] if "-" in x else x))
    )
)

In [ ]:
# further harmonise date formats and extract years
cholera_outbreaks["year"] = cholera_outbreaks["year"].str.pad(width=3, side="left", fillchar="0")
cholera_outbreaks["year"] = cholera_outbreaks["year"].str.pad(width=4, side="left", fillchar="2")
cholera_outbreaks["year"] = cholera_outbreaks["year"].astype(np.int64)

In [ ]:
# check years
sorted(cholera_outbreaks["year"].unique())

In [ ]:
# check case where there seems to be a typo
cholera_outbreaks.loc[cholera_outbreaks["year"] == 2019]

In [ ]:
# fix typo
cholera_outbreaks.loc[
    (cholera_outbreaks["file_name"] == "39th_2015.pdf") & (cholera_outbreaks["year"] == 2019),
    "year",
] = 2015

In [ ]:
# harmonise date formats to extract month
cholera_outbreaks["month"] = cholera_outbreaks["start_date_clean"].apply(
    lambda x: (
        x.split(".")[1]
        if "." in x
        else (x.split("/")[1] if "/" in x else (x.split("-")[1] if "-" in x else x))
    )
)

In [ ]:
# cast month to integer
cholera_outbreaks["month"] = cholera_outbreaks["month"].astype(np.int64)

In [ ]:
# check months
sorted(cholera_outbreaks["month"].unique())

In [ ]:
cholera_outbreaks.shape

In [ ]:
# drop rows, drop columns, add cholera outbreak column
cholera_outbreaks = cholera_outbreaks.loc[
    cholera_outbreaks["year"].isin(np.arange(2010, 2019))
].reset_index(drop=True)
cholera_outbreaks = cholera_outbreaks.drop(["disease", "cases", "deaths", "start_date"], axis=1)
cholera_outbreaks = cholera_outbreaks.rename(columns={"start_date_clean": "start_date"})
cholera_outbreaks["outbreak"] = 1
cholera_outbreaks.shape

In [ ]:
cholera_outbreaks.head()

## Map states, districts and location

Next, we need to map states and districts to their geographic location. We do this with the help of the [Database of Global Administrative Areas](https://www.gadm.org/index.html). We download the Level 2 administrative zones for India and explore them before we extract the relevant information and merge it with the outbreaks data.

In [ ]:
!wget --recursive --no-directories --no-clobber --directory-prefix=../../../data/outbreaks https://biogeo.ucdavis.edu/data/gadm3.6/shp/gadm36_IND_shp.zip

In [ ]:
!unzip -u -d ../../../data/outbreaks/gadm36_IND_shp ../../../data/outbreaks/gadm36_IND_shp.zip

In [ ]:
india = gpd.read_file("../../../data/outbreaks/gadm36_IND_shp/gadm36_IND_2.shp")
india.shape

In [ ]:
india.crs

In [ ]:
india.info()

In [ ]:
india.head()

In [ ]:
india.plot()

In [ ]:
# select states, districts and geometry
states_districts = india[["NAME_1", "NAME_2", "geometry"]].copy()

In [ ]:
# rename columns
states_districts.columns = ["state", "district", "geometry"]

In [ ]:
# make states and districts lowercase to simplify the mapping
states_districts["state"] = states_districts["state"].str.lower()
states_districts["district"] = states_districts["district"].str.lower()

In [ ]:
# create list with unique districts
districts = states_districts["district"].unique().tolist()

In [ ]:
# harmonise district names in district to simplify mapping
cholera_outbreaks.loc[cholera_outbreaks["district"] == "mahabubnagar", "district"] = "mahbubnagar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "ahmedabad", "district"] = "ahmadabad"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "howrah", "district"] = "haora"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "hooghly", "district"] = "hugli"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "hoogly", "district"] = "hugli"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "villupuram", "district"] = "viluppuram"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "haridwar", "district"] = "hardwar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "davangere", "district"] = "davanagere"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "davengere", "district"] = "davanagere"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "davangare", "district"] = "davanagere"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "tiruchirapalli", "district"] = (
    "tiruchirappalli"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "darang", "district"] = "darrang"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "virudhunager", "district"] = "virudunagar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "chikkaballapur", "district"] = (
    "chikballapura"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "gondia", "district"] = "gondiya"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "purulia", "district"] = "puruliya"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kalaburagi", "district"] = "gulbarga"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kalburgi", "district"] = "gulbarga"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "gulburga", "district"] = "gulbarga"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "berhampur", "district"] = "ganjam"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "sholapur", "district"] = "solapur"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "raigad", "district"] = "raigarh"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "panchmahal", "district"] = "panch mahals"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "sibsagar", "district"] = "sivasagar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "banaskantha", "district"] = "banas kantha"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "chamarajnagar", "district"] = "chamrajnagar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "khargaon", "district"] = "west nimar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "mysuru", "district"] = "mysore"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "mohali", "district"] = (
    "sahibzada ajit singh nagar"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "delhi", "district"] = "west"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kawardha", "district"] = "kabeerdham"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "chirtadurga", "district"] = "chitradurga"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "budgam", "district"] = "badgam"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "jangir", "district"] = "janjgir-champa"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kancheepuramsaidapet", "district"] = (
    "kancheepuram"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "thiruvannamalai", "district"] = (
    "tiruvannamalai"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "sabarkantha", "district"] = "sabar kantha"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "alwar, sikar, jaipur", "district"] = "alwar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "badwani", "district"] = "barwani"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "balasore", "district"] = "baleshwar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "bangalore urban", "district"] = "bangalore"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "bardhaman", "district"] = "barddhaman"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "belagavi", "district"] = "belgaum"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "chota udaipur", "district"] = (
    "chhota udaipur"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "d&n haveli", "district"] = (
    "dadra and nagar haveli"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "dadra & nagar haveli", "district"] = (
    "dadra and nagar haveli"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "east champaran", "district"] = (
    "purba champaran"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "gandhi nagar", "district"] = "gandhinagar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "janjgir", "district"] = "janjgir-champa"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kalaburgi (gulbarga)", "district"] = (
    "gulbarga"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kamrup ( metro)", "district"] = (
    "kamrup metropolitan"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kamrup metro", "district"] = (
    "kamrup metropolitan"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "kancheepuram(saidapet)", "district"] = (
    "kancheepuram"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "khandwa", "district"] = "east nimar"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "malda", "district"] = "maldah"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "medinipur west", "district"] = (
    "pashchim medinipur"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "name of district mathura", "district"] = (
    "mathura"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "new delhi", "district"] = "west"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "north 24 paraganas", "district"] = (
    "north 24 parganas"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "north 24 pargans", "district"] = (
    "north 24 parganas"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "north 24 praganas", "district"] = (
    "north 24 parganas"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "north delhi", "district"] = "west"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "north district", "district"] = "west"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "purlia", "district"] = "puruliya"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "rampurhat", "district"] = "birbhum"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "ratlam sonepur", "district"] = "ratlam"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "sas nagar", "district"] = (
    "sahibzada ajit singh nagar"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "south west district", "district"] = "west"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "uttar kannada", "district"] = (
    "uttara kannada"
)
cholera_outbreaks.loc[cholera_outbreaks["district"] == "west delhi", "district"] = "west"
cholera_outbreaks.loc[cholera_outbreaks["district"] == "yamuna nagar", "district"] = "yamunanagar"

In [ ]:
# map states and geometry to districts
cholera_outbreaks_mapped = pd.merge(
    states_districts, cholera_outbreaks, how="right", on="district"
)[["state", "district", "start_date", "year", "month", "outbreak", "geometry"]].reset_index(
    drop=True
)
cholera_outbreaks_mapped.shape

In [ ]:
# find districts with identical names appearing in different states
duplicate_districts = (
    states_districts["district"]
    .value_counts()[states_districts["district"].value_counts() > 1]
    .index.tolist()
)
cholera_outbreaks_mapped[
    cholera_outbreaks_mapped["district"].isin(duplicate_districts)
].sort_values("district")

We need to check to which state the duplicated districts actually belong.

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "aurangabad")
    & (cholera_outbreaks_mapped["month"] == 11)
]  # maharashtra

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "aurangabad")
    & (cholera_outbreaks_mapped["month"] == 7)
]  # maharashtra

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "bijapur") & (cholera_outbreaks_mapped["month"] == 8)
]  # karnataka

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "bijapur") & (cholera_outbreaks_mapped["month"] == 7)
]  # karnataka

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "bijapur") & (cholera_outbreaks_mapped["month"] == 8)
]  # karnataka

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "pratapgarh")
    & (cholera_outbreaks_mapped["month"] == 7)
]  # rajasthan

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "raigarh") & (cholera_outbreaks_mapped["month"] == 7)
]  # maharashtra

In [ ]:
cholera_outbreaks_mapped.loc[
    (cholera_outbreaks_mapped["district"] == "raigarh") & (cholera_outbreaks_mapped["month"] == 3)
]  # maharashtra

We figured out to which state the duplicated districts actually belong and drop the rows that contain wrongly mapped data.

In [ ]:
cholera_outbreaks_mapped = cholera_outbreaks_mapped.drop(
    cholera_outbreaks_mapped.loc[
        (cholera_outbreaks_mapped["state"].isin(["bihar", "chhattisgarh", "uttar pradesh"]))
        & (
            cholera_outbreaks_mapped["district"].isin(
                ["aurangabad", "bijapur", "raigarh", "pratapgarh"]
            )
        )
    ].index
)
cholera_outbreaks_mapped.shape

We also need to drop duplicated cholera outbreaks. Some cholera outbreaks are followed up on in PDF files of later weeks and therefore appear multiple times in the extracted data.

In [ ]:
cholera_outbreaks_mapped[cholera_outbreaks_mapped.duplicated()].sort_values(
    ["state", "district", "start_date"]
).shape

In [ ]:
cholera_outbreaks_mapped = (
    cholera_outbreaks_mapped.drop_duplicates().drop("start_date", axis=1).reset_index(drop=True)
)
cholera_outbreaks_mapped.shape

Finally, we save the mapped and deduplicated cholera outbreaks in shapefile format.

In [ ]:
%%time

cholera_outbreaks_mapped.to_file(
    "../../../data/outbreaks/monthly_cholera_outbreaks_india_district_2010_2018.shp"
)